# CS 228 experiment: defense variants (hands-off)

**Inputs (right panel -> Add Input):** the dataset `celeba-spoof-for-face-antispoofing` (Datasets tab) AND your
notebook `cs228-data` (Notebooks tab), which holds the prebuilt `data_pad2`. Without `cs228-data` it rebuilds the
data (+40 min). Session options: GPU T4 x2, Internet On. Then **Save Version -> Save & Run All (Commit)**.
About 3 hours (3 variants). Results: Versions -> newest -> Output (`bundle3.zip`) and the log.

In [ ]:
# 1. Get the code
%cd /kaggle/working
!rm -rf cs228-pad-project
GITHUB_USER = "ShivamShelar101"
!git clone https://github.com/{GITHUB_USER}/cs228-pad-project.git
%cd cs228-pad-project
!pip install -q onnxscript matplotlib
!git log --oneline -1

In [ ]:
# 2. Dataset location (attach it first: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing)
import glob, os
hits = []
for depth in range(1, 4):   # shallow search only: never walk the 750k-file dataset
    hits += glob.glob("/kaggle/input/" + "*/" * (depth - 1) + "celeba-spoof-for-face-antispoofing")
print("found:", hits)
assert hits, "Dataset not attached: Add Input -> Datasets -> celeba-spoof-for-face-antispoofing"
SRC = hits[0]
print("dataset at:", SRC)

In [ ]:
# 3. Data: reuse the prebuilt data_pad2 from the attached cs228-data notebook, else build it
found = []
for depth in range(1, 6):
    found += glob.glob("/kaggle/input/" + "*/" * (depth - 1) + "data_pad2")
print("prebuilt data:", found)
if found:
    !ln -sfn {found[0]} ./data_pad2
else:
    !python -m data.prepare_celeba_spoof --src {SRC} --dst ./data_pad2 --per_class 3000 --test_per_class 600 --max_per_subject 8
!for s in train val test; do for c in live spoof; do echo $s/$c $(ls data_pad2/$s/$c | wc -l); done; done

In [ ]:
# 4. Baseline (no defense)
!python -m training.train_baseline --data_root ./data_pad2 --epochs 5 --seed 0 --out checkpoints/baseline

In [ ]:
# 5a. Variant E2: adversarial training at a SMALL budget (2/255). Faint spoof cues cannot be fully erased.
!python -m training.train_hardened --data_root ./data_pad2 --epochs 10 --lr 3e-5 --seed 0 --eps 0.00784 --out checkpoints/eps2

In [ ]:
# 5b. Variant E4: medium budget (4/255)
!python -m training.train_hardened --data_root ./data_pad2 --epochs 10 --lr 3e-5 --seed 0 --eps 0.01569 --out checkpoints/eps4

In [ ]:
# 5c. Variant F3: old budget (8/255), attacked FAKES count 3x in the loss
!python -m training.train_hardened --data_root ./data_pad2 --epochs 10 --lr 3e-5 --seed 0 --fake_adv_weight 3 --out checkpoints/fake3

In [ ]:
# 6. Test-set evaluation with an attack-strength sweep (1-16/255), plus report
B = '--baseline_glob "checkpoints/baseline/seed*_best.pt"'
for v in ["eps2", "eps4", "fake3"]:
    !python -m evaluation.evaluate --data_root ./data_pad2 --sweep 1 2 4 8 16 {B} --hardened_glob "checkpoints/{v}/seed*_best.pt" --out results/eval_{v}.json
!python -m evaluation.make_report --run "E2: trained at 2/255=results/eval_eps2.json" --run "E4: trained at 4/255=results/eval_eps4.json" --run "F3: 8/255, fakes x3=results/eval_fake3.json" --out results/report3
print(open("results/report3.md").read())

In [ ]:
# 7. Did the models even learn their OWN training fakes? (1000 training images, with the same sweep)
for v in ["eps2", "eps4", "fake3"]:
    !python -m evaluation.evaluate --data_root ./data_pad2 --split train --limit 1000 --sweep 1 2 4 8 {B} --hardened_glob "checkpoints/{v}/seed*_best.pt" --out results/train_check_{v}.json
!python -m evaluation.make_report --run "E2 on TRAIN=results/train_check_eps2.json" --run "E4 on TRAIN=results/train_check_eps4.json" --run "F3 on TRAIN=results/train_check_fake3.json" --out results/train_check
print(open("results/train_check.md").read())

In [ ]:
# 8. Which kinds of fakes each model accepts (clean images)
for ck in ["baseline", "eps2", "eps4", "fake3"]:
    !python -m evaluation.by_spoof_type --src {SRC} --data_root ./data_pad2 --ckpt checkpoints/{ck}/seed0_best.pt

In [ ]:
# 9. Pack the results
!zip -qr bundle3.zip results checkpoints/baseline/seed0_best.pt checkpoints/eps2/seed0_best.pt checkpoints/eps4/seed0_best.pt checkpoints/fake3/seed0_best.pt
!ls -lh bundle3.zip